### Imports

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras import layers, models

In [7]:
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU devices: []


### Paths and settings

In [8]:
TRAIN_DIR = Path("../dataset/split/train")
VAL_DIR = Path("../dataset/split/val")
TEST_DIR = Path("../dataset/split/test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

### Load datasets

In [9]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

Found 6238 files belonging to 8 classes.
Found 1337 files belonging to 8 classes.
Found 1337 files belonging to 8 classes.


### Check class names

In [10]:
class_names = train_ds.class_names

print("Class names:")
for index, class_name in enumerate(class_names):
    print(index, "->", class_name)

Class names:
0 -> CCI_Caterpillars
1 -> CCI_Leaflets
2 -> Gray Leaf Spot
3 -> Healthy_Leaves
4 -> Leaf Rot
5 -> WCLWD_DryingofLeaflets
6 -> WCLWD_Flaccidity
7 -> WCLWD_Yellowing


### Rebuild class weights safely

In [11]:
weight_lookup = {
    "CCI_Caterpillars": 1.125180,
    "CCI_Leaflets": 1.402428,
    "Gray Leaf Spot": 0.522621,
    "Healthy_Leaves": 9.066860,
    "Leaf Rot": 0.678634,
    "WCLWD_DryingofLeaflets": 1.032781,
    "WCLWD_Flaccidity": 1.042447,
    "WCLWD_Yellowing": 1.027339,
}

class_weights = {
    index: weight_lookup[class_name]
    for index, class_name in enumerate(class_names)
}

class_weights

{0: 1.12518,
 1: 1.402428,
 2: 0.522621,
 3: 9.06686,
 4: 0.678634,
 5: 1.032781,
 6: 1.042447,
 7: 1.027339}

### Improve input pipeline performance

In [12]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

### Data augmentation

In [13]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

### Custom CNN baseline

In [14]:
num_classes = len(class_names)

baseline_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(
        num_classes,
        activation="softmax"
    ),
])

baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,472 (1.49 MB)

 Trainable params: 390,472 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

### Add callbacks

In [16]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
    ),
]

### Train baseline

In [17]:
EPOCHS = 20

In [ ]:
baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
) 

Epoch 1/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 574s 3s/step - accuracy: 0.4032 - loss: 1.5150 - val_accuracy: 0.5288 - val_loss: 1.0860 - learning_rate: 0.0010
Epoch 2/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 534s 3s/step - accuracy: 0.5529 - loss: 1.0877 - val_accuracy: 0.5610 - val_loss: 1.0961 - learning_rate: 0.0010
Epoch 3/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 591s 3s/step - accuracy: 0.6271 - loss: 0.8856 - val_accuracy: 0.6051 - val_loss: 0.9063 - learning_rate: 0.0010
Epoch 4/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 594s 3s/step - accuracy: 0.6722 - loss: 0.7627 - val_accuracy: 0.7105 - val_loss: 0.7678 - learning_rate: 0.0010
Epoch 5/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 559s 3s/step - accuracy: 0.7405 - loss: 0.6408 - val_accuracy: 0.6904 - val_loss: 0.7506 - learning_rate: 0.0010
Epoch 6/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 576s 3s/step - accuracy: 0.7873 - loss: 0.5268 - val_accuracy: 0.8437 - val_loss: 0.4030 - learning_rate: 0.0010
Epoch 7/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 555s 3s/step - accuracy: 0.8245 - loss: 0.

In [19]:
baseline_model.save("../backend/trained_models/custom_cnn_baseline.keras")